# 序列建模与 Seq2Seq 架构企业笔试手撕通关宝典
> **面向对象**：互联网大厂/AI 独角兽企业 NLP / 机器翻译 / 序列生成算法岗面试手撕  
> **核心涵盖**：经典 Encoder-Decoder、Reverse 梯度压缩、Peeky 上下文旁路、Bahdanau 加性注意力、Luong 乘性注意力、Teacher Forcing 暴露偏差、Scheduled Sampling 计划采样、Beam Search 束搜索解码、Corpus-BLEU 评测  
> **设计准则**：纯 PyTorch / NumPy 极简实现，零第三方黑盒包，透彻剖析信息瓶颈、时序对齐与自回归推理。

---
### 核心模块速览
1. **模块一**：经典 Encoder-Decoder LSTM 架构手撕 (信息瓶颈 Bottleneck Vector 传递)
2. **模块二**：Reverse Trick 机制与时序梯度几何距离压缩验证
3. **模块三**：Peeky Seq2Seq 架构手撕 (多步特征旁路无损直连)
4. **模块四**：Bahdanau 加性注意力机制手撕 (Additive Attention)
5. **模块五**：Luong 乘性注意力机制手撕 (Multiplicative Attention)
6. **模块六**：Teacher Forcing 机制与暴露偏差 (Exposure Bias) 剖析
7. **模块七**：Scheduled Sampling 计划采样策略手撕
8. **模块八**：Beam Search 束搜索解码器手撕 (带长度惩罚 Length Penalty)
9. **模块九**：机器翻译评测 BLEU 分数矩阵级手撕 (N-gram 匹配与简短惩罚 BP)

---
## 模块一：经典 Encoder-Decoder LSTM 架构手撕 (信息瓶颈 Bottleneck Vector 传递)

### 【笔试考点与陷阱】
1. **信息瓶颈 (Information Bottleneck)**：编码器将源端变长序列压缩为单一固定维度的末时刻隐状态 $(h_T, c_T)$，长文本时丢失早前上下文；
2. **状态交接**：解码器的初始隐状态由编码器的最后一步隐状态无缝承接：$(h_0^{\text{dec}}, c_0^{\text{dec}}) = (h_T^{\text{enc}}, c_T^{\text{enc}})$。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class ClassicEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

    def forward(self, x):
        # x: (B, src_len)
        embeds = self.embedding(x)             # (B, src_len, embed_dim)
        outputs, (h_n, c_n) = self.lstm(embeds) # outputs: (B, src_len, H), h_n: (1, B, H)
        return outputs, (h_n, c_n)

class ClassicDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, y, prev_state):
        # y: (B, tgt_len), prev_state: (h, c)
        embeds = self.embedding(y)              # (B, tgt_len, embed_dim)
        outputs, (h_n, c_n) = self.lstm(embeds, prev_state)
        logits = self.fc_out(outputs)           # (B, tgt_len, vocab_size)
        return logits, (h_n, c_n)

# 快速验证
src_x = torch.randint(0, 50, (2, 5)) # B=2, src_len=5
tgt_y = torch.randint(0, 50, (2, 4)) # B=2, tgt_len=4
encoder = ClassicEncoder(50, 16, 32)
decoder = ClassicDecoder(50, 16, 32)

enc_outs, (h, c) = encoder(src_x)
logits, (h_dec, c_dec) = decoder(tgt_y, (h, c))
print("经典 Seq2Seq 解码 Logits 形状 (B, tgt_len, V):", logits.shape)
assert logits.shape == (2, 4, 50)

---
## 模块二：Reverse Trick 机制与反向时序梯度距离压缩验证

### 【笔试考点与物理本质】
1. **正序输入距离**：源序列首词 $x_1$ 传到目标序列首词 $y_1$，梯度需要穿透 $T_{\text{src}} + 1$ 步；
2. **逆序输入 (Reverse)**：$x_1$ 位于输入序列末端，与 $y_1$ 物理交互距离直接缩短为 **$1$ 步**！
3. **笔试代码写法**：极其简单高雅，仅需一行索引切片：`x_reversed = torch.flip(x, dims=[1])` 或 `x[:, ::-1]`。

In [ ]:
def reverse_input_tensor(x):
    """
    将输入序列在时序维度进行逆序翻转 (Reverse Trick)
    x: (B, seq_len)
    """
    return torch.flip(x, dims=[1])

# 测试验证 Reverse 距离压缩效果
x_seq = torch.tensor([[10, 20, 30, 40, 50]])
x_rev = reverse_input_tensor(x_seq)
print("正序序列:", x_seq.numpy())
print("逆序序列:", x_rev.numpy())
assert x_rev[0, 0] == 50 and x_rev[0, -1] == 10
print(">>> Reverse Trick 成功将首词距离由 6 步直降为 1 步！")

---
## 模块三：Peeky Seq2Seq 架构手撕 (多步特征旁路无损直连)

### 【笔试考点与架构细节】
1. **打破单步传递瓶颈**：传统解码器只有在 $t=0$ 步接收一次 $h_T^{\text{enc}}$；
2. **Peeky（窥视机制）**：将编码器的末状态 $h_T$ 复制并在每一个时间步与当前输入 $y_t$ 拼接，同时还直连到最后的输出预测层：
   $$\tilde{y}_t = [\text{Embed}(y_t) ; h_T], \quad \text{Logit}_t = W [h_t^{\text{dec}} ; h_T] + b$$

In [ ]:
class PeekyDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        # 解码器 LSTM 的输入维度扩充为 embed_dim + hidden_dim (直连拼接)
        self.lstm = nn.LSTM(embed_dim + hidden_dim, hidden_dim, batch_first=True)
        # 输出分类器输入扩充为 hidden_dim + hidden_dim
        self.fc_out = nn.Linear(hidden_dim + hidden_dim, vocab_size)

    def forward(self, y, h_enc, c_enc):
        B, tgt_len = y.shape
        embeds = self.embedding(y)              # (B, tgt_len, embed_dim)
        
        # 将 h_enc: (1, B, H) 变为 (B, 1, H) 并广播拓展为 (B, tgt_len, H)
        h_enc_expanded = h_enc.squeeze(0).unsqueeze(1).repeat(1, tgt_len, 1)
        
        # 旁路拼接输入
        lstm_input = torch.cat([embeds, h_enc_expanded], dim=-1) # (B, tgt_len, embed_dim + H)
        outputs, (h_n, c_n) = self.lstm(lstm_input, (h_enc, c_enc))
        
        # 旁路直连输出层
        classifier_input = torch.cat([outputs, h_enc_expanded], dim=-1) # (B, tgt_len, 2H)
        logits = self.fc_out(classifier_input)
        return logits, (h_n, c_n)

peeky_dec = PeekyDecoder(50, 16, 32)
p_logits, _ = peeky_dec(tgt_y, h, c)
print("Peeky 解码 Logits 形状:", p_logits.shape)
assert p_logits.shape == (2, 4, 50)
print(">>> Peeky 全时间步直连通道构建验证成功！")

---
## 模块四：Bahdanau 加性注意力机制手撕 (Additive Attention)

### 【笔试考点与公式推导】
- **打分函数 (Score Function)**：
  $$e_{t, i} = v_a^T \tanh(W_a s_{t-1} + U_a h_i)$$
  其中 $s_{t-1}$ 为解码器上一时刻隐状态，$h_i$ 为编码器全部隐状态。
- **权重归一化**：$\alpha_{t, i} = \text{softmax}(e_{t, i})$；
- **上下文向量**：$c_t = \sum_i \alpha_{t, i} h_i$。

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, enc_hidden_dim, dec_hidden_dim, attn_dim):
        super().__init__()
        self.W_dec = nn.Linear(dec_hidden_dim, attn_dim, bias=False)
        self.W_enc = nn.Linear(enc_hidden_dim, attn_dim, bias=False)
        self.v_a = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, dec_hidden, enc_outputs):
        """
        dec_hidden: (B, dec_hidden_dim) 上一时刻解码器隐状态
        enc_outputs: (B, src_len, enc_hidden_dim) 编码器全部时刻状态
        """
        # 投影计算
        proj_dec = self.W_dec(dec_hidden).unsqueeze(1)          # (B, 1, attn_dim)
        proj_enc = self.W_enc(enc_outputs)                     # (B, src_len, attn_dim)
        
        # 加性融合 + tanh
        energy = torch.tanh(proj_dec + proj_enc)                # (B, src_len, attn_dim)
        scores = self.v_a(energy).squeeze(-1)                   # (B, src_len)
        
        # Softmax 归一化注意力权重
        weights = F.softmax(scores, dim=-1)                     # (B, src_len)
        
        # 加权求和得到上下文向量 c_t
        context = torch.bmm(weights.unsqueeze(1), enc_outputs).squeeze(1) # (B, enc_hidden_dim)
        return context, weights

# 测试验证
bahdanau = BahdanauAttention(enc_hidden_dim=32, dec_hidden_dim=32, attn_dim=16)
dummy_dec_h = torch.randn(2, 32)
ctx, weights = bahdanau(dummy_dec_h, enc_outs)
print("Bahdanau 注意力上下文向量形状 (B, H):", ctx.shape)
print("注意力权重总和 (验证 Softmax=1):", weights.sum(dim=-1).detach().numpy())
assert np.allclose(weights.sum(dim=-1).detach().numpy(), [1.0, 1.0])

---
## 模块五：Luong 乘性注意力机制手撕 (Multiplicative Attention)

### 【笔试考点与差异对比】
- **计算时序**：Bahdanau 用 $s_{t-1}$，Luong 采用当前步解码状态 $s_t$；
- **打分公式**：
  $$\text{Score}(s_t, h_i) = s_t^T W_a h_i \quad (\text{General模式})$$
- 乘性注意力可以通过高效的 Batch 矩阵乘法 (`torch.bmm`) 并行计算，计算吞吐显著优于加性注意力。

In [ ]:
class LuongAttention(nn.Module):
    def __init__(self, enc_hidden_dim, dec_hidden_dim):
        super().__init__()
        self.W_a = nn.Linear(enc_hidden_dim, dec_hidden_dim, bias=False)

    def forward(self, dec_hidden, enc_outputs):
        """
        dec_hidden: (B, dec_hidden_dim)
        enc_outputs: (B, src_len, enc_hidden_dim)
        """
        # 线性映射编码器状态: (B, src_len, dec_hidden_dim)
        proj_enc = self.W_a(enc_outputs)
        
        # 点乘打分: (B, 1, dec_hidden_dim) bmm (B, dec_hidden_dim, src_len) -> (B, 1, src_len)
        scores = torch.bmm(dec_hidden.unsqueeze(1), proj_enc.transpose(1, 2)).squeeze(1)
        weights = F.softmax(scores, dim=-1)                     # (B, src_len)
        context = torch.bmm(weights.unsqueeze(1), enc_outputs).squeeze(1)
        return context, weights

luong = LuongAttention(enc_hidden_dim=32, dec_hidden_dim=32)
ctx_l, weights_l = luong(dummy_dec_h, enc_outs)
print("Luong 乘性注意力上下文向量形状:", ctx_l.shape)
assert ctx_l.shape == (2, 32)
print(">>> Luong Attention 验证通过！")

---
## 模块六：Teacher Forcing 机制与暴露偏差 (Exposure Bias) 剖析

### 【笔试考点与面试话术】
1. **Teacher Forcing**：训练时无论上一时刻模型预测对错，强制将**真实标签 (Ground Truth)** 作为下一时刻解码器输入，防止前期梯度震荡；
2. **暴露偏差 (Exposure Bias)**：测试时模型只能自力更生以自身预测输入（Free Running）。若某一步预测错误，误差会如滚雪球般在时序中累加放大崩溃！

In [ ]:
def teacher_forcing_step(decoder, prev_token, prev_hidden, ground_truth_next, tf_ratio=1.0):
    """
    单步自回归结合 Teacher Forcing
    tf_ratio=1.0: 纯 Teacher Forcing (输入黄金标准)
    tf_ratio=0.0: 纯 Free Running (输入自身上一轮 Argmax)
    """
    # forward 单步
    logits, next_hidden = decoder(prev_token.unsqueeze(1), prev_hidden)
    pred_token = torch.argmax(logits.squeeze(1), dim=-1)
    
    use_tf = (torch.rand(1).item() < tf_ratio)
    next_input_token = ground_truth_next if use_tf else pred_token
    return logits, next_hidden, next_input_token

# 测试模拟
dummy_tok = torch.tensor([1, 2])
dummy_gt = torch.tensor([10, 11])
logits_step, h_step, next_tok = teacher_forcing_step(decoder, dummy_tok, (h, c), dummy_gt, tf_ratio=1.0)
assert torch.equal(next_tok, dummy_gt)
print("Teacher Forcing 成功注入黄金标准:", next_tok.numpy())

---
## 模块七：Scheduled Sampling 计划采样策略手撕

### 【笔试考点与衰减策略】
- **破局暴露偏差**：训练初期以 $100\%$ 概率使用 Teacher Forcing，随训练 Epoch 增加，逐渐按**线性衰减、指数衰减或反 S 型曲线**降低 $P_{\text{tf}}$，让模型平滑过渡到自回归生成。

In [ ]:
class ScheduledSampler:
    def __init__(self, initial_prob=1.0, min_prob=0.1, decay_rate=0.05, mode="linear"):
        self.prob = initial_prob
        self.min_prob = min_prob
        self.decay_rate = decay_rate
        self.mode = mode

    def step(self, epoch):
        if self.mode == "linear":
            self.prob = max(self.min_prob, 1.0 - epoch * self.decay_rate)
        elif self.mode == "exponential":
            self.prob = max(self.min_prob, (1.0 - self.decay_rate) ** epoch)
        return self.prob

sampler = ScheduledSampler(initial_prob=1.0, min_prob=0.1, decay_rate=0.1)
probs = [sampler.step(epoch) for epoch in range(12)]
print("Scheduled Sampling 采样概率随 Epoch 平滑衰减曲线:")
for ep, p in enumerate(probs[:6]):
    print(f"  Epoch {ep}: Teacher Forcing 概率 = {p:.2f}")
assert probs[-1] == 0.1

---
## 模块八：Beam Search 束搜索解码器手撕 (带长度惩罚 Length Penalty)

### 【笔试高频硬核考点】
1. **贪心 vs. 束搜索**：贪心每步只保留 Top-1 易陷入局部最优；束搜索维护大小为 $B$ (Beam Width) 的候选假说池；
2. **长度惩罚 (Length Penalty)**：累加对数概率 $\sum \log p$ 会无条件偏向极短句子。必须引入惩罚因子：
   $$\text{Score} = \frac{\sum_{t=1}^L \log p_t}{\left( \frac{5 + L}{6} \right)^\alpha}, \quad \alpha \in [0.6, 0.8]$$

In [ ]:
def beam_search_decoder(step_predict_fn, start_token, end_token, beam_width=3, max_steps=10, alpha=0.7):
    """
    通用纯 Python 束搜索解码器
    step_predict_fn(token, state) -> (log_probs (vocab_size,), next_state)
    """
    # 候选假说池: 元素为 (cum_log_prob, tokens_list, state)
    beams = [(0.0, [start_token], None)]
    completed_hypotheses = []

    for _ in range(max_steps):
        all_candidates = []
        for cum_prob, tokens, state in beams:
            last_tok = tokens[-1]
            if last_tok == end_token:
                # 已经生成完终止符，移入完成集合
                lp = ((5.0 + len(tokens)) / 6.0) ** alpha
                completed_hypotheses.append((cum_prob / lp, tokens))
                continue
                
            log_probs, next_state = step_predict_fn(last_tok, state)
            # 挑出 Top-B 个最优词
            topk_probs, topk_indices = torch.topk(log_probs, beam_width)
            for p, idx in zip(topk_probs.tolist(), topk_indices.tolist()):
                all_candidates.append((cum_prob + p, tokens + [idx], next_state))
                
        if not all_candidates:
            break
            
        # 排序并截断保留 Top-B
        all_candidates.sort(key=lambda x: x[0], reverse=True)
        beams = all_candidates[:beam_width]

    if completed_hypotheses:
        completed_hypotheses.sort(key=lambda x: x[0], reverse=True)
        return completed_hypotheses[0]
    else:
        lp = ((5.0 + len(beams[0][1])) / 6.0) ** alpha
        return (beams[0][0] / lp, beams[0][1])

# 构造模拟单步预测测试
def mock_predict_fn(token, state):
    # 随机构造 10 个词的对数概率
    log_probs = F.log_softmax(torch.tensor([0.1, 0.5, 0.2, 0.05, 0.05, 2.0, 0.1, 0.1, 0.8, 0.1]), dim=-1)
    return log_probs, None

best_score, best_seq = beam_search_decoder(mock_predict_fn, start_token=0, end_token=9, beam_width=3, max_steps=5)
print("Beam Search 解码出的最佳得分与序列:", round(best_score, 4), best_seq)
assert len(best_seq) > 1

---
## 模块九：机器翻译评测 BLEU 分数矩阵级手撕 (Corpus-BLEU)

### 【笔试考点】
1. **修正 N-gram 精度 (Modified N-gram Precision)**：截断统计防止单词重复作弊；
2. **简短惩罚 (Brevity Penalty, BP)**：
   $$\text{BP} = \begin{cases} 1 & \text{if } c > r \\ \exp(1 - r/c) & \text{if } c \le r \end{cases}$$
3. **最终公式**：$\text{BLEU} = \text{BP} \cdot \exp\left( \sum_{n=1}^N w_n \log p_n \right)$。

In [ ]:
import collections
import math

def compute_bleu(candidate_tokens, reference_tokens, max_n=4, weights=(0.25, 0.25, 0.25, 0.25)):
    """
    计算句子级 BLEU-4 评分
    """
    c = len(candidate_tokens)
    r = len(reference_tokens)
    
    # 1. 简短惩罚 BP
    if c == 0:
        return 0.0
    bp = 1.0 if c > r else math.exp(1.0 - float(r) / c)
    
    # 2. 逐阶统计 N-gram 精度
    precisions = []
    for n in range(1, max_n + 1):
        cand_ngrams = collections.Counter([tuple(candidate_tokens[i:i+n]) for i in range(len(candidate_tokens) - n + 1)])
        ref_ngrams = collections.Counter([tuple(reference_tokens[i:i+n]) for i in range(len(reference_tokens) - n + 1)])
        
        # 截断匹配计数
        match_count = 0
        for ng, count in cand_ngrams.items():
            match_count += min(count, ref_ngrams.get(ng, 0))
            
        total_cand_ngrams = max(1, len(candidate_tokens) - n + 1)
        # 避免平滑为 0 导致 log(0)
        p_n = (match_count + 1e-8) / total_cand_ngrams
        precisions.append(p_n)
        
    # 3. 几何加权平均
    s = sum(w * math.log(p) for w, p in zip(weights, precisions))
    bleu = bp * math.exp(s)
    return bleu

# 测试验证
cand = ["the", "cat", "sat", "on", "the", "mat"]
ref = ["the", "cat", "is", "on", "the", "mat"]
score = compute_bleu(cand, ref)
print(f"BLEU-4 评分: {score * 100:.2f}")
assert 0.0 < score <= 1.0
print(">>> BLEU-4 向量化计算校验成功！")

---
## 企业笔试手撕核心口诀与雷区速记卡

```
1. 经典瓶颈死穴: h_T 承载全句记忆越长越崩，Reverse 一行翻转首词距离直降为 1。
2. Peeky 全步直连: 隐状态在每个时间步和分类器输出端双重拼接，小模型必备神技。
3. Bahdanau vs Luong: 加性注意力用 s_{t-1} + h_i 并 tanh，乘性注意力用 s_t^T W h_i 可 bmm 并行。
4. 暴露偏差与采样: TF 训得爽测时滚雪球，Scheduled Sampling 随 Epoch 线性衰减训练鲁棒性。
5. 束搜索防短惩罚: 累加对数概率天然偏向短句，除以 ((5+L)/6)^alpha 才能选出高质量长译文。
```